In [1]:
import joblib
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Cargamos el modelo y el vectorizador guardados en el notebook 02
best_rf = joblib.load('../models/random_forest_toxic_model.pkl')
tfidf = joblib.load('../models/tfidf_vectorizer.pkl')

lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(text):
    tokens = word_tokenize(text)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmatized_tokens

def predecir_toxicidad(comentario):
    tokens = tokenize_and_lemmatize(comentario)
    tokens_text = ' '.join(tokens)
    vector = tfidf.transform([tokens_text])
    prediccion = best_rf.predict(vector)[0]
    return "Tóxico" if prediccion else "No tóxico"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
comentarios_prueba = [
    # Esperados como tóxicos
    ("She is a brat.", "Tóxico"),
    ("kick her pompous ass", "Tóxico"),
    ("What a spoiled child", "Tóxico"),
    ("Go back to school, little bitch.", "Tóxico"),
    ("Fuck off to school.", "Tóxico"),
    # Esperados como no tóxicos
    ("Goodbye to my home in Bahrain", "No tóxico"),
    ("RIP Florida", "No tóxico"),
    ("Australia is covered by sea, but Melbourne is totally safe", "No tóxico"),
    ("How can we stop this from happening", "No tóxico"),
    ("I wouldn't mind if humans got extinct to be honest.", "No tóxico"),
]

aciertos = 0
for comentario, esperado in comentarios_prueba:
    resultado = predecir_toxicidad(comentario)
    acierto = "✓" if resultado == esperado else "✗"
    if resultado == esperado:
        aciertos += 1
    print(f"{acierto} '{comentario}' -> Predicho: {resultado} | Esperado: {esperado}")

print(f"\nAciertos: {aciertos}/{len(comentarios_prueba)}")

✗ 'She is a brat.' -> Predicho: No tóxico | Esperado: Tóxico
✗ 'kick her pompous ass' -> Predicho: No tóxico | Esperado: Tóxico
✗ 'What a spoiled child' -> Predicho: No tóxico | Esperado: Tóxico
✓ 'Go back to school, little bitch.' -> Predicho: Tóxico | Esperado: Tóxico
✓ 'Fuck off to school.' -> Predicho: Tóxico | Esperado: Tóxico
✓ 'Goodbye to my home in Bahrain' -> Predicho: No tóxico | Esperado: No tóxico
✓ 'RIP Florida' -> Predicho: No tóxico | Esperado: No tóxico
✓ 'Australia is covered by sea, but Melbourne is totally safe' -> Predicho: No tóxico | Esperado: No tóxico
✓ 'How can we stop this from happening' -> Predicho: No tóxico | Esperado: No tóxico
✓ 'I wouldn't mind if humans got extinct to be honest.' -> Predicho: No tóxico | Esperado: No tóxico

Aciertos: 7/10


## Conclusiones de la evaluación

- Evaluación sobre 10 comentarios reales de YouTube (fuente: Sage Journals) con
  etiqueta esperada conocida: 7/10 aciertos (70%).
- El modelo detecta correctamente el 100% de los comentarios no tóxicos (5/5).
- De los 5 comentarios tóxicos, solo detecta 2/5 (40%): los que contienen
  vocabulario explícitamente ofensivo ("bitch", "fuck off").
- Falla en toxicidad implícita o sin palabrotas evidentes ("She is a brat.",
  "kick her pompous ass", "What a spoiled child"), donde el insulto depende del
  contexto y no de términos malsonantes reconocibles por TF-IDF.
- Esto es coherente con el recall de 0.61 obtenido en el notebook 02 para la
  clase tóxica: el modelo tiene buena precisión detectando toxicidad explícita,
  pero limitada capacidad para detectar sarcasmo o desprecio sutil sin vocabulario
  ofensivo directo — una limitación esperable dado que TF-IDF no captura contexto
  semántico ni tono.